# 다국어 EU 법령 분류 임베딩 및 전처리

* **데이터**: MultiEURLEX 8개 언어 (`EN`, `DE`, `FR`, `EL`, `FI`, `PL`, `BG`, `MT`), Level 1 (21개 라벨)
* **분할**: `main_train` 10,000 / `adaptation_pool` 4,604 / `validation` 5,000 / `test` 5,000 (동일 CELEX)
* **임베딩**: `intfloat/multilingual-e5-small` (frozen, `query:` prefix, 480-token chunk, overlap 0, mean pooling, L2 정규화)
* **TF-IDF**: NFKC, 소문자화, 숫자 `0` 치환, 공백 정리, `min_df=5`, `max_df=0.5`, `sublinear_tf=True`, `(1,1)` / `(1,2)`
* **교차언어 정렬 (RQ4)**: 타겟 언어 test → 영어 test 코사인 최근접 이웃(Top-1) 검색 및 동일 CELEX 코사인 유사도

In [ ]:
# 0. 환경 설정 및 데이터 준비
import os
import io
import re
import time
import json
import math
import shutil
import hashlib
import unicodedata
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer as SklearnTfidfVectorizer

try:
    import cupy as cp
    import cudf
    import cuml
    from cuml.neighbors import NearestNeighbors as cuNearestNeighbors
    from cuml.linear_model import LogisticRegression as cuLogisticRegression
    from cuml.feature_extraction.text import TfidfVectorizer as cuTfidfVectorizer
    HAS_RAPIDS = True
    print(f"RAPIDS enabled: cudf={cudf.__version__}, cuml={cuml.__version__}, cupy={cp.__version__}")
except ImportError:
    HAS_RAPIDS = False
    print("RAPIDS not found. Using PyTorch/NumPy fallback.")

try:
    from google.colab import drive, auth
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

LANGUAGES = ["en", "de", "fr", "el", "fi", "pl", "bg", "mt"]
TARGET_LANGUAGES = ["de", "fr", "el", "fi", "pl", "bg", "mt"]
SPLITS = ["train", "validation", "test"]

MODEL_NAME = "intfloat/multilingual-e5-small"
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

PARQUET_FILE_IDS: Dict[str, str] = {
    "en_train.parquet": "1MNHTW1PxQ54KJD4Tvkk733GcXV8g4Dsc",
    "en_validation.parquet": "1UJCjLZrgR9uGbYqFYj-fBzIkGeWOB0Xj",
    "en_test.parquet": "1hbzkBPZrl0RT876omnyapdTjY-EedNoy",
    "de_train.parquet": "1THdhSGbsu_Vy-Mw_qcik4hCo5xzjakZ_",
    "de_validation.parquet": "1RIuCbwMprgdrglqIe5ttTVvzzRO2uuGj",
    "de_test.parquet": "1dM8eAHnFTR2QfqsILltoTvfCwciCjRqg",
    "fr_train.parquet": "1w3U-sfe1bnMFpdBBEvipAigpFbeR363o",
    "fr_validation.parquet": "1d3wVvu2ZgrbmaWQjciga4ssFmXEoOPaa",
    "fr_test.parquet": "1ltUkHFZHapTWagX0kf-PVwVlW2CaxYu8",
    "el_train.parquet": "1L7_Hj_uOL6ST8dVpUmVtRceXmIWEy76S",
    "el_validation.parquet": "1HccvR4ohm-5PPCYvSnTjv-PiAoLxM3h5",
    "el_test.parquet": "1YzYTQ1yZzR_Y8nGqEZIUXudTOuB7iPDp",
    "fi_train.parquet": "18H0CGgRBerfHXNfuxyTtA4MQsY2jHhnu",
    "fi_validation.parquet": "1sN9c55wwKlzaNX1QAzSea-4cxMKBdFci",
    "fi_test.parquet": "1sivXdyoN7_Mdd7iEbyU54Fpouinku76U",
    "pl_train.parquet": "1kgG13P6fYI1oQ6nLBKUMQB84_ub-LgEN",
    "pl_validation.parquet": "1yQC9ei6Puw6JQr7VcrmQSKTT9DBJmZGy",
    "pl_test.parquet": "1DoxpHJ2Vk2kn-NDwRcW4J2pOZ4r_XoOb",
    "bg_train.parquet": "1lDGJYblrwCmgIqjGYBrnW7VV4oLIR77c",
    "bg_validation.parquet": "1iIR1JFvLXWuLd46Mlt5rFvUTPWV-HQ5p",
    "bg_test.parquet": "1W871jCFWvoFLHiuxuSKFmS3EH2E2Y4pk",
    "mt_train.parquet": "1-y2CaVngupZ9mBVUubrtoFtovotAlg3l",
    "mt_validation.parquet": "1s0_LUkfEsSin0aU2659E6Pfe0myGUeE3",
    "mt_test.parquet": "1bhjhAS757ZhnWmLPCJuTmp1F6DOhetkT",
}

CANDIDATE_DATA_DIRS = [
    Path("/content/local_data"),
    Path("/content/drive/MyDrive/기계학습/data"),
    Path("/content/drive/.shortcut-targets-by-id/1yHEq5aNdbhY50dqgj963aBcEoQDOY_Pz/data"),
]
DATA_DIR = next(
    (p for p in CANDIDATE_DATA_DIRS if p.exists() and (p / "en_test.parquet").exists()),
    Path("/content/local_data"),
)
DATA_DIR.mkdir(parents=True, exist_ok=True)


def ensure_all_24_parquet_files(target_dir: Path = DATA_DIR):
    missing = [fname for fname in PARQUET_FILE_IDS if not (target_dir / fname).exists()]
    if not missing:
        print(f"All 24 parquet files ready: {target_dir}")
        return
    print(f"Downloading {len(missing)} parquet files to {target_dir}...")
    if IN_COLAB:
        auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    service = build("drive", "v3")
    for fname in tqdm(missing, desc="Downloading parquet files"):
        file_id = PARQUET_FILE_IDS[fname]
        out_path = target_dir / fname
        request = service.files().get_media(fileId=file_id)
        with open(out_path, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request, chunksize=32 * 1024 * 1024)
            done = False
            while not done:
                _, done = downloader.next_chunk()
    print(f"Download complete: {target_dir}")


ensure_all_24_parquet_files(DATA_DIR)

WORK_DIR = Path("/content/drive/MyDrive/기계학습_산출물") if IN_COLAB else Path("./outputs")
DIRS = {
    "splits": WORK_DIR / "01_splits",
    "stats": WORK_DIR / "02_tokenization_stats",
    "emb_core": WORK_DIR / "02_embeddings_core_480_mean",
    "emb_p2_first": WORK_DIR / "02_embeddings_p2_first_chunk",
    "emb_p3": WORK_DIR / "02_embeddings_p3_extensions",
    "tfidf_rq1": WORK_DIR / "04_tfidf_rq1",
    "rq4": WORK_DIR / "07_rq4_alignment",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

[INFO] RAPIDS 활성화 완료! cudf=26.06.00, cuml=26.06.00, cupy=14.0.1
Mounted at /content/drive
[INFO] '/content/local_data'에 24개 Parquet 파일이 없어 Google Drive API로 직접 다운로드합니다...


[SUCCESS] 24개 Parquet 파일 다운로드 완료: /content/local_data


In [ ]:
# 1. Parquet 로더 및 CELEX 분할 로드
def detect_columns(cols) -> Tuple[str, str, str]:
    id_candidates = ["celex_id", "id", "doc_id", "celex"]
    text_candidates = ["text", "document", "body", "content"]
    label_candidates = ["labels", "eurovoc_concepts", "level_1_labels", "label"]
    col_list = list(cols)
    id_col = next((c for c in id_candidates if c in col_list), col_list[0])
    text_col = next((c for c in text_candidates if c in col_list), col_list[1])
    label_col = next((c for c in label_candidates if c in col_list), col_list[-1])
    return id_col, text_col, label_col


def load_lang_split_texts_and_word_counts(
    lang: str,
    split: str,
    ordered_celex_list: List[str],
    data_dir: Path = DATA_DIR,
) -> Tuple[List[str], np.ndarray]:
    """Parquet 파일에서 CELEX 순서에 맞춰 텍스트와 공백 기준 어절 수를 반환합니다."""
    fpath = data_dir / f"{lang}_{split}.parquet"
    if HAS_RAPIDS:
        gdf = cudf.read_parquet(fpath)
        id_col, text_col, _ = detect_columns(gdf.columns)
        gdf = gdf[[id_col, text_col]].rename(columns={id_col: "celex_id", text_col: "text"})
        gdf["celex_id"] = gdf["celex_id"].astype("str")
        gdf["text"] = gdf["text"].fillna("")

        pdf = gdf.to_pandas().set_index("celex_id").loc[ordered_celex_list]
        texts = pdf["text"].tolist()

        s_gpu = cudf.Series(texts)
        word_counts = s_gpu.str.token_count().to_numpy().astype(np.int32)
        return texts, word_counts
    else:
        df = pd.read_parquet(fpath)
        id_col, text_col, _ = detect_columns(df.columns)
        df = df[[id_col, text_col]].rename(columns={id_col: "celex_id", text_col: "text"})
        df["celex_id"] = df["celex_id"].astype(str)
        df["text"] = df["text"].fillna("").astype(str)
        df = df.set_index("celex_id").loc[ordered_celex_list]
        texts = df["text"].tolist()
        word_counts = np.array([len(t.split()) for t in texts], dtype=np.int32)
        return texts, word_counts


split_json_path = DIRS["splits"] / "celex_splits.json"
assert split_json_path.exists(), f"Missing file: {split_json_path}"
with open(split_json_path, "r", encoding="utf-8") as f:
    SPLIT_INFO = json.load(f)

print(
    f"Loaded CELEX splits:\n"
    f"  - main_train: {len(SPLIT_INFO['main_train']):,}\n"
    f"  - adaptation_pool: {len(SPLIT_INFO['adaptation_pool']):,}\n"
    f"  - validation: {len(SPLIT_INFO['validation']):,}\n"
    f"  - test: {len(SPLIT_INFO['test']):,}"
)

[SUCCESS] 기존 CELEX 스플릿 JSON 즉시 로드 완료!
  - main_train: 10,000개
  - adaptation_pool: 4,604개
  - validation: 5,000개
  - test: 5,000개


In [ ]:
# 2. 토큰화 및 480-token 청킹
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.model_max_length = 10_000_000

QUERY_PREFIX_IDS: List[int] = tokenizer.encode("query:", add_special_tokens=False)
CLS_ID: int = tokenizer.cls_token_id if tokenizer.cls_token_id is not None else tokenizer.bos_token_id
SEP_ID: int = tokenizer.sep_token_id if tokenizer.sep_token_id is not None else tokenizer.eos_token_id
PAD_ID: int = tokenizer.pad_token_id
UNK_ID: int = tokenizer.unk_token_id

PREFIX_ARR = np.array([CLS_ID] + QUERY_PREFIX_IDS, dtype=np.int32)
SEP_ARR = np.array([SEP_ID], dtype=np.int32)
PREFIX_LEN = len(QUERY_PREFIX_IDS)

print(f"Tokenizer loaded (is_fast={tokenizer.is_fast}), prefix IDs={QUERY_PREFIX_IDS}")


def batch_tokenize_and_chunk_documents(
    texts: List[str],
    word_counts: np.ndarray,
    celex_ids: List[str],
    chunk_size: int = 480,
    overlap: int = 0,
    max_chunks: Optional[int] = None,
    tok_batch_size: int = 2000,
) -> Tuple[List[np.ndarray], np.ndarray, pd.DataFrame]:
    """문서 배치 토큰화 후 480-token 단위 청크, 문서 인덱스, 문서별 통계 DataFrame을 반환합니다."""
    n_docs = len(texts)
    all_raw_ids: List[List[int]] = []
    for start in range(0, n_docs, tok_batch_size):
        batch_enc = tokenizer(
            texts[start : start + tok_batch_size],
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
        )["input_ids"]
        all_raw_ids.extend(batch_enc)

    flat_chunks: List[np.ndarray] = []
    chunk_doc_indices: List[int] = []

    num_tokens_arr = np.zeros(n_docs, dtype=np.int32)
    num_chunks_arr = np.zeros(n_docs, dtype=np.int32)
    num_unk_arr = np.zeros(n_docs, dtype=np.int32)
    exceeds_512_arr = np.zeros(n_docs, dtype=bool)

    step = max(1, chunk_size - overlap)
    empty_chunk = np.concatenate([PREFIX_ARR, SEP_ARR])

    for doc_idx, raw_ids in enumerate(all_raw_ids):
        n_tok = len(raw_ids)
        num_tokens_arr[doc_idx] = n_tok

        if n_tok == 0:
            flat_chunks.append(empty_chunk)
            chunk_doc_indices.append(doc_idx)
            num_chunks_arr[doc_idx] = 1
            continue

        arr_ids = np.asarray(raw_ids, dtype=np.int32)
        num_unk_arr[doc_idx] = int(np.sum(arr_ids == UNK_ID))
        exceeds_512_arr[doc_idx] = (n_tok + PREFIX_LEN + 2) > 512

        c_cnt = 0
        for s_idx in range(0, n_tok, step):
            piece = arr_ids[s_idx : s_idx + chunk_size]
            if len(piece) == 0:
                break
            chunk_seq = np.concatenate([PREFIX_ARR, piece, SEP_ARR])
            flat_chunks.append(chunk_seq)
            chunk_doc_indices.append(doc_idx)
            c_cnt += 1
            if max_chunks is not None and c_cnt >= max_chunks:
                break
            if s_idx + chunk_size >= n_tok:
                break
        num_chunks_arr[doc_idx] = c_cnt

    fertility_arr = np.where(word_counts > 0, num_tokens_arr / np.maximum(1, word_counts), np.nan)
    df_doc_stats = pd.DataFrame({
        "celex_id": celex_ids,
        "num_words": word_counts,
        "num_tokens": num_tokens_arr,
        "num_chunks": num_chunks_arr,
        "num_unk": num_unk_arr,
        "fertility": fertility_arr,
        "exceeds_512": exceeds_512_arr,
    })

    return flat_chunks, np.asarray(chunk_doc_indices, dtype=np.int64), df_doc_stats


EXPECTED_TEST_CHUNKS = {
    "en": 5.75, "de": 6.42, "fr": 7.02, "pl": 6.75,
    "bg": 6.48, "el": 7.81, "fi": 6.64, "mt": 10.91,
}

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

[INFO] FastTokenizer(is_fast=True) 로드 완료! Prefix IDs=[41, 1294, 12]


In [ ]:
# 3. 청크 인코딩 및 문서 임베딩 집계 (Mean Pooling + L2 정규화)
@torch.inference_mode()
def encode_and_pool_documents_gpu_fast(
    model: AutoModel,
    flat_chunks: List[np.ndarray],
    chunk_doc_indices: np.ndarray,
    n_docs: int,
    batch_size: int = 256,
    device: str = DEVICE,
    compute_p2_first_chunk: bool = True,
    compute_p3_maxpool: bool = True,
) -> Tuple[np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
    """청크 인코딩 후 문서 단위 mean pooling(Core), first-chunk(P2), max-pooling(P3) 임베딩을 반환합니다."""
    n_chunks = len(flat_chunks)
    lengths = np.array([len(c) for c in flat_chunks], dtype=np.int32)
    order = np.argsort(lengths)
    sorted_doc_indices = chunk_doc_indices[order]

    _, first_chunk_flat_positions = np.unique(chunk_doc_indices, return_index=True)

    hidden_dim = model.config.hidden_size
    doc_sum_gpu = torch.zeros((n_docs, hidden_dim), dtype=torch.float32, device=device)
    doc_count_gpu = torch.zeros((n_docs, 1), dtype=torch.float32, device=device)

    all_chunk_vecs_sorted = (
        torch.empty((n_chunks, hidden_dim), dtype=torch.float32, device="cpu")
        if (compute_p2_first_chunk or compute_p3_maxpool)
        else None
    )

    use_amp = (device == "cuda")

    for b_start in tqdm(range(0, n_chunks, batch_size), desc="Encoding chunks", leave=False):
        b_end = min(b_start + batch_size, n_chunks)
        b_idx = order[b_start:b_end]
        b_len = int(lengths[b_idx[-1]])
        bsz = b_end - b_start

        input_np = np.full((bsz, b_len), PAD_ID, dtype=np.int64)
        mask_np = np.zeros((bsz, b_len), dtype=np.int64)
        for i, orig_i in enumerate(b_idx):
            seq = flat_chunks[orig_i]
            L = len(seq)
            input_np[i, :L] = seq
            mask_np[i, :L] = 1

        if device == "cuda":
            input_ids = torch.from_numpy(input_np).pin_memory().to(device, non_blocking=True)
            attention_mask = torch.from_numpy(mask_np).pin_memory().to(device, non_blocking=True)
        else:
            input_ids = torch.from_numpy(input_np)
            attention_mask = torch.from_numpy(mask_np)

        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden = outputs.last_hidden_state.float()

        mask_f = attention_mask.unsqueeze(-1).float()
        chunk_vecs = (last_hidden * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-9)

        batch_doc_ids = torch.from_numpy(sorted_doc_indices[b_start:b_end]).to(device, non_blocking=True)
        doc_sum_gpu.index_add_(0, batch_doc_ids, chunk_vecs)
        ones = torch.ones((bsz, 1), dtype=torch.float32, device=device)
        doc_count_gpu.index_add_(0, batch_doc_ids, ones)

        if all_chunk_vecs_sorted is not None:
            all_chunk_vecs_sorted[b_start:b_end] = chunk_vecs.cpu()

    doc_mean_gpu = doc_sum_gpu / doc_count_gpu.clamp(min=1.0)
    core_emb = F.normalize(doc_mean_gpu, p=2, dim=1).cpu().numpy().astype(np.float32)

    first_emb = None
    max_emb = None
    if all_chunk_vecs_sorted is not None:
        inv_order = np.argsort(order)
        all_chunk_vecs = all_chunk_vecs_sorted[inv_order]

        if compute_p2_first_chunk:
            first_vecs = all_chunk_vecs[first_chunk_flat_positions]
            first_emb = F.normalize(first_vecs, p=2, dim=1).numpy().astype(np.float32)

        if compute_p3_maxpool:
            max_list = []
            boundaries = np.r_[first_chunk_flat_positions, n_chunks]
            for d_i in range(n_docs):
                s_c, e_c = boundaries[d_i], boundaries[d_i + 1]
                max_list.append(all_chunk_vecs[s_c:e_c].max(dim=0).values)
            max_vecs = torch.stack(max_list, dim=0)
            max_emb = F.normalize(max_vecs, p=2, dim=1).numpy().astype(np.float32)

    return core_emb, first_emb, max_emb

In [ ]:
# 4. 8개 언어 임베딩 추출 및 토큰화 통계 저장
def run_all_core_embeddings_fast(batch_size: int = 256) -> pd.DataFrame:
    print(f"Loading model: {MODEL_NAME} on {DEVICE}")
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()
    for param in model.parameters():
        param.requires_grad = False

    summary_rows = []

    for lang in LANGUAGES:
        for split_name, raw_split_key in [
            ("main_train", "train"),
            ("adaptation_pool", "train"),
            ("validation", "validation"),
            ("test", "test"),
        ]:
            t0 = time.time()
            celex_list = SPLIT_INFO[split_name]
            core_out_path = DIRS["emb_core"] / f"{lang}_{split_name}.npy"
            first_out_path = DIRS["emb_p2_first"] / f"{lang}_{split_name}_first_chunk.npy"
            maxpool_out_path = DIRS["emb_p3"] / f"{lang}_{split_name}_maxpool.npy"
            celex_out_path = DIRS["emb_core"] / f"{lang}_{split_name}_celex.json"
            doc_stats_path = DIRS["stats"] / f"{lang}_{split_name}_doc_stats.parquet"

            save_extra = (split_name in ["main_train", "validation", "test"])

            if core_out_path.exists() and doc_stats_path.exists():
                df_stats = pd.read_parquet(doc_stats_path)
                print(f"[Cached] {lang.upper()} - {split_name} ({len(df_stats):,} docs)")
            else:
                texts, word_counts = load_lang_split_texts_and_word_counts(lang, raw_split_key, celex_list)
                flat_chunks, chunk_doc_indices, df_stats = batch_tokenize_and_chunk_documents(
                    texts=texts,
                    word_counts=word_counts,
                    celex_ids=celex_list,
                    chunk_size=480,
                    overlap=0,
                )
                print(
                    f"[{lang.upper()} - {split_name}] docs={len(texts):,}, "
                    f"chunks={len(flat_chunks):,} (mean={df_stats['num_chunks'].mean():.2f})"
                )

                core_emb, first_emb, max_emb = encode_and_pool_documents_gpu_fast(
                    model=model,
                    flat_chunks=flat_chunks,
                    chunk_doc_indices=chunk_doc_indices,
                    n_docs=len(texts),
                    batch_size=batch_size,
                    device=DEVICE,
                    compute_p2_first_chunk=save_extra,
                    compute_p3_maxpool=save_extra,
                )

                np.save(core_out_path, core_emb)
                with open(celex_out_path, "w", encoding="utf-8") as f:
                    json.dump(celex_list, f)
                if save_extra and first_emb is not None:
                    np.save(first_out_path, first_emb)
                if save_extra and max_emb is not None:
                    np.save(maxpool_out_path, max_emb)

                df_stats.to_parquet(doc_stats_path, index=False)
                print(f"  saved ({time.time() - t0:.1f}s) | shape={core_emb.shape}")

            total_tokens = df_stats["num_tokens"].sum()
            total_words = df_stats["num_words"].sum()
            total_unk = df_stats["num_unk"].sum()

            summary_rows.append({
                "lang": lang.upper(),
                "split": split_name,
                "n_docs": len(df_stats),
                "pct_over_512": round(df_stats["exceeds_512"].mean() * 100, 2),
                "mean_chunks": round(df_stats["num_chunks"].mean(), 2),
                "expected_test_chunks": EXPECTED_TEST_CHUNKS.get(lang, np.nan) if split_name == "test" else np.nan,
                "max_chunks": int(df_stats["num_chunks"].max()),
                "mean_tokens": round(df_stats["num_tokens"].mean(), 1),
                "mean_words": round(df_stats["num_words"].mean(), 1),
                "corpus_fertility": round(total_tokens / max(1, total_words), 4),
                "doc_mean_fertility": round(df_stats["fertility"].mean(), 4),
                "unk_rate": round(total_unk / max(1, total_tokens), 6),
            })

    df_summary = pd.DataFrame(summary_rows)
    summary_csv_path = DIRS["stats"] / "tokenization_and_chunk_summary_8langs.csv"
    df_summary.to_csv(summary_csv_path, index=False)
    print(f"Saved summary: {summary_csv_path}")
    return df_summary


df_tok_summary = run_all_core_embeddings_fast(batch_size=256)
display(df_tok_summary[df_tok_summary["split"] == "test"])

[INFO] Loading frozen feature extractor: intfloat/multilingual-e5-small on cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[CACHED] EN - main_train (10,000 docs) 이미 존재함
[CACHED] EN - adaptation_pool (4,604 docs) 이미 존재함
[CACHED] EN - validation (5,000 docs) 이미 존재함
[CACHED] EN - test (5,000 docs) 이미 존재함
[CACHED] DE - main_train (10,000 docs) 이미 존재함
[CACHED] DE - adaptation_pool (4,604 docs) 이미 존재함
[CACHED] DE - validation (5,000 docs) 이미 존재함
[CACHED] DE - test (5,000 docs) 이미 존재함
[CACHED] FR - main_train (10,000 docs) 이미 존재함
[CACHED] FR - adaptation_pool (4,604 docs) 이미 존재함
[CACHED] FR - validation (5,000 docs) 이미 존재함
[CACHED] FR - test (5,000 docs) 이미 존재함
[CACHED] EL - main_train (10,000 docs) 이미 존재함
[CACHED] EL - adaptation_pool (4,604 docs) 이미 존재함
[CACHED] EL - validation (5,000 docs) 이미 존재함
[CACHED] EL - test (5,000 docs) 이미 존재함
[CACHED] FI - main_train (10,000 docs) 이미 존재함
[CACHED] FI - adaptation_pool (4,604 docs) 이미 존재함
[CACHED] FI - validation (5,000 docs) 이미 존재함
[CACHED] FI - test (5,000 docs) 이미 존재함
[CACHED] PL - main_train (10,000 docs) 이미 존재함
[CACHED] PL - adaptation_pool (4,604 docs) 이미 존재함
[PL 

  GPU Encoding Chunks:   0%|          | 0/97 [00:00<?, ?it/s]

... (16 outputs left out)

  -> 완료 및 Drive 저장 성공! (150.4초 소요) | Shape=(5000, 384)
[MT - test] Docs=5,000 | Total Chunks=54,537 (Mean=10.91) -> GPU 인코딩 시작...


  GPU Encoding Chunks:   0%|          | 0/214 [00:00<?, ?it/s]

  -> 완료 및 Drive 저장 성공! (215.0초 소요) | Shape=(5000, 384)

[SUCCESS] 8개 언어 토큰화 통계 저장 완료: /content/drive/MyDrive/기계학습_산출물/02_tokenization_stats/tokenization_and_chunk_summary_8langs.csv

=== [검증] Test 스플릿 언어별 평균 청크 수 vs 설계서 기준치 ===


,lang,split,n_docs,pct_over_512,mean_chunks,expected_test_chunks,max_chunks,mean_tokens,mean_words,corpus_fertility,doc_mean_fertility,unk_rate
3,EN,test,5000,67.78,5.75,5.75,574,2549.1,1753.8,1.4535,1.5242,0.0
7,DE,test,5000,72.16,6.42,6.42,630,2857.7,1599.6,1.7865,1.8214,0.0
11,FR,test,5000,75.24,7.02,7.02,715,3140.8,1920.8,1.6351,1.6736,0.0
15,EL,test,5000,91.60,7.81,7.81,807,3500.6,1819.7,1.9237,2.0212,0.0
19,FI,test,5000,75.78,6.64,6.64,679,2932.4,1216.0,2.4115,2.4273,0.0
23,PL,test,5000,78.56,6.75,6.75,694,2980.6,1550.8,1.9219,1.9945,0.0
27,BG,test,5000,74.10,6.48,6.48,648,2891.6,1732.2,1.6693,1.7060,0.0
31,MT,test,5000,98.24,10.91,10.91,1142,5007.4,1430.8,3.4997,3.5911,0.0


In [ ]:
# 5. TF-IDF 전처리 및 희소 행렬 저장 (RQ1)
DIGIT_REGEX = re.compile(r"\d")
WHITESPACE_REGEX = re.compile(r"\s+")


def batch_preprocess_texts_tfidf(texts: List[str]) -> List[str]:
    """NFKC 정규화, 소문자화, 숫자 0 치환, 공백 정리를 수행합니다."""
    nfkc_texts = [unicodedata.normalize("NFKC", t) for t in texts]
    if HAS_RAPIDS:
        gs = cudf.Series(nfkc_texts)
        gs = gs.str.lower()
        gs = gs.str.replace(r"\d", "0", regex=True)
        gs = gs.str.normalize_spaces()
        return gs.to_arrow().to_pylist()
    else:
        return [WHITESPACE_REGEX.sub(" ", DIGIT_REGEX.sub("0", t.lower())).strip() for t in nfkc_texts]


def run_and_cache_all_tfidf():
    tfidf_meta_rows = []
    for lang in LANGUAGES:
        t0 = time.time()
        tr_texts, _ = load_lang_split_texts_and_word_counts(lang, "train", SPLIT_INFO["main_train"])
        va_texts, _ = load_lang_split_texts_and_word_counts(lang, "validation", SPLIT_INFO["validation"])
        te_texts, _ = load_lang_split_texts_and_word_counts(lang, "test", SPLIT_INFO["test"])

        train_clean = batch_preprocess_texts_tfidf(tr_texts)
        val_clean = batch_preprocess_texts_tfidf(va_texts)
        test_clean = batch_preprocess_texts_tfidf(te_texts)

        for ngram_tag, ngram_range in [("unigram", (1, 1)), ("bigram", (1, 2))]:
            out_tr = DIRS["tfidf_rq1"] / f"{lang}_main_train_{ngram_tag}.npz"
            out_va = DIRS["tfidf_rq1"] / f"{lang}_validation_{ngram_tag}.npz"
            out_te = DIRS["tfidf_rq1"] / f"{lang}_test_{ngram_tag}.npz"

            if out_tr.exists() and out_va.exists() and out_te.exists():
                X_tr = sp.load_npz(out_tr)
                vocab_size = X_tr.shape[1]
            else:
                vec = SklearnTfidfVectorizer(
                    ngram_range=ngram_range,
                    min_df=5,
                    max_df=0.5,
                    sublinear_tf=True,
                    lowercase=False,
                    stop_words=None,
                    dtype=np.float32,
                )
                X_tr = vec.fit_transform(train_clean)
                X_va = vec.transform(val_clean)
                X_te = vec.transform(test_clean)

                sp.save_npz(out_tr, X_tr)
                sp.save_npz(out_va, X_va)
                sp.save_npz(out_te, X_te)
                vocab_size = len(vec.vocabulary_)

            tfidf_meta_rows.append({
                "lang": lang.upper(),
                "ngram": ngram_tag,
                "vocab_size": vocab_size,
                "train_shape": str((10000, vocab_size)),
            })
        print(f"[TF-IDF] {lang.upper()} done ({time.time() - t0:.1f}s)")

    df_tfidf_meta = pd.DataFrame(tfidf_meta_rows)
    df_tfidf_meta.to_csv(DIRS["tfidf_rq1"] / "tfidf_vocab_summary.csv", index=False)
    return df_tfidf_meta


df_tfidf_summary = run_and_cache_all_tfidf()
display(df_tfidf_summary)

[TF-IDF Cached] EN (unigram & bigram) 완료 (91.3초)
[TF-IDF Cached] DE (unigram & bigram) 완료 (97.7초)
[TF-IDF Cached] FR (unigram & bigram) 완료 (106.4초)
[TF-IDF Cached] EL (unigram & bigram) 완료 (140.0초)
[TF-IDF Cached] FI (unigram & bigram) 완료 (90.8초)
[TF-IDF Cached] PL (unigram & bigram) 완료 (98.3초)
[TF-IDF Cached] BG (unigram & bigram) 완료 (124.4초)
[TF-IDF Cached] MT (unigram & bigram) 완료 (102.1초)


,lang,ngram,vocab_size,train_shape
0,EN,unigram,14648,"(10000, 14648)"
1,EN,bigram,176723,"(10000, 176723)"
2,DE,unigram,34018,"(10000, 34018)"
3,DE,bigram,221599,"(10000, 221599)"
4,FR,unigram,19362,"(10000, 19362)"
5,FR,bigram,196370,"(10000, 196370)"
6,EL,unigram,32328,"(10000, 32328)"
7,EL,bigram,230410,"(10000, 230410)"
8,FI,unigram,55310,"(10000, 55310)"
9,FI,bigram,233873,"(10000, 233873)"


In [ ]:
# 6. 교차언어 정렬(Alignment) 및 설명변수 계산 (RQ4)
def compute_rq4_alignment_rapids(include_p3_mrr: bool = True) -> pd.DataFrame:
    """타겟 언어 test 5,000개 → 영어 test 5,000개 코사인 Top-1 검색 정확도 및 동일 CELEX 코사인 평균을 계산합니다."""
    en_test_path = DIRS["emb_core"] / "en_test.npy"
    assert en_test_path.exists(), "Missing en_test.npy"

    emb_en = np.load(en_test_path)
    n_test = emb_en.shape[0]
    gold_indices = np.arange(n_test)

    if HAS_RAPIDS:
        knn_gpu = cuNearestNeighbors(n_neighbors=1, metric="cosine")
        knn_gpu.fit(emb_en)
        emb_en_gpu = cp.asarray(emb_en)

    summary_csv_path = DIRS["stats"] / "tokenization_and_chunk_summary_8langs.csv"
    df_tok = pd.read_csv(summary_csv_path)
    df_tok_test = df_tok[df_tok["split"] == "test"].set_index("lang")

    rq4_rows = []
    for lang in TARGET_LANGUAGES:
        emb_tg = np.load(DIRS["emb_core"] / f"{lang}_test.npy")

        if HAS_RAPIDS:
            distances, indices = knn_gpu.kneighbors(emb_tg)
            top1_pred = cp.asnumpy(indices).reshape(-1)
            top1_acc = float((top1_pred == gold_indices).mean())

            emb_tg_gpu = cp.asarray(emb_tg)
            sim_matrix_gpu = emb_tg_gpu @ emb_en_gpu.T
            diag_gpu = cp.diag(sim_matrix_gpu)
            same_celex_cos_mean = float(diag_gpu.mean().get())
            same_celex_cos_sd = float(diag_gpu.std().get())

            mrr_val = np.nan
            if include_p3_mrr:
                ranks_gpu = (sim_matrix_gpu > diag_gpu[:, None]).sum(axis=1) + 1
                mrr_val = float((1.0 / ranks_gpu).mean().get())
        else:
            sim_matrix = emb_tg @ emb_en.T
            same_celex_cos_mean = float(np.diag(sim_matrix).mean())
            same_celex_cos_sd = float(np.diag(sim_matrix).std())
            top1_pred = np.argmax(sim_matrix, axis=1)
            top1_acc = float((top1_pred == gold_indices).mean())
            mrr_val = np.nan
            if include_p3_mrr:
                diag_scores = np.diag(sim_matrix)[:, None]
                ranks = (sim_matrix > diag_scores).sum(axis=1) + 1
                mrr_val = float((1.0 / ranks).mean())

        tok_row = df_tok_test.loc[lang.upper()]
        rq4_rows.append({
            "lang": lang.upper(),
            "top1_retrieval_acc": round(top1_acc, 4),
            "same_celex_cosine_mean": round(same_celex_cos_mean, 4),
            "same_celex_cosine_sd": round(same_celex_cos_sd, 4),
            "p3_alignment_mrr": round(mrr_val, 4) if not np.isnan(mrr_val) else np.nan,
            "mean_chunks": tok_row["mean_chunks"],
            "mean_tokens": tok_row["mean_tokens"],
            "mean_words": tok_row["mean_words"],
            "corpus_fertility": tok_row["corpus_fertility"],
            "doc_mean_fertility": tok_row["doc_mean_fertility"],
        })

    df_rq4 = pd.DataFrame(rq4_rows)
    rq4_out_path = DIRS["rq4"] / "rq4_alignment_and_fertility_table.csv"
    df_rq4.to_csv(rq4_out_path, index=False)
    print(f"Saved RQ4 table: {rq4_out_path}")
    return df_rq4


df_rq4_vars = compute_rq4_alignment_rapids()
display(df_rq4_vars)

[SUCCESS] cuML/CuPy 기반 RQ4 Alignment 및 설명변수 테이블 저장 완료: /content/drive/MyDrive/기계학습_산출물/07_rq4_alignment/rq4_alignment_and_fertility_table.csv


,lang,top1_retrieval_acc,same_celex_cosine_mean,same_celex_cosine_sd,p3_alignment_mrr,mean_chunks,mean_tokens,mean_words,corpus_fertility,doc_mean_fertility
0,DE,0.7440,0.9254,0.0127,0.8082,6.42,2857.7,1599.6,1.7865,1.8214
1,FR,0.7588,0.9451,0.0106,0.8082,7.02,3140.8,1920.8,1.6351,1.6736
2,EL,0.4144,0.9157,0.0151,0.4926,7.81,3500.6,1819.7,1.9237,2.0212
3,FI,0.4492,0.9062,0.0148,0.5264,6.64,2932.4,1216.0,2.4115,2.4273
4,PL,0.6002,0.9206,0.0140,0.6615,6.75,2980.6,1550.8,1.9219,1.9945
5,BG,0.6608,0.9211,0.0129,0.7273,6.48,2891.6,1732.2,1.6693,1.7060
6,MT,0.5276,0.9358,0.0106,0.6127,10.91,5007.4,1430.8,3.4997,3.5911


In [ ]:
# 7. OvR 로지스틱 회귀 분류기 함수
def fit_predict_ovr_logreg_cuml(
    X_train: np.ndarray,
    Y_train: np.ndarray,
    X_eval: np.ndarray,
    C: float = 10.0,
    max_iter: int = 2000,
) -> np.ndarray:
    """21개 라벨에 대해 OvR 로지스틱 회귀(L2, max_iter=2000) 학습 후 예측 확률을 반환합니다."""
    n_labels = Y_train.shape[1]
    probs = np.zeros((X_eval.shape[0], n_labels), dtype=np.float32)

    for j in range(n_labels):
        y_col = Y_train[:, j].astype(np.int32)
        if y_col.sum() == 0 or y_col.sum() == len(y_col):
            probs[:, j] = float(y_col.mean())
            continue
        if HAS_RAPIDS:
            clf = cuLogisticRegression(penalty="l2", C=float(C), max_iter=max_iter, tol=1e-4)
            clf.fit(X_train, y_col)
            p = clf.predict_proba(X_eval)
            probs[:, j] = cp.asnumpy(p)[:, 1] if hasattr(p, "get") else np.asarray(p)[:, 1]
        else:
            from sklearn.linear_model import LogisticRegression
            clf = LogisticRegression(penalty="l2", solver="lbfgs", C=float(C), max_iter=max_iter)
            clf.fit(X_train, y_col)
            probs[:, j] = clf.predict_proba(X_eval)[:, 1]
    return probs

In [ ]:
# 8. 산출물 검증 및 압축
def verify_and_package_all_outputs(create_zip: bool = True) -> Dict:
    manifest = {}
    for key, folder in DIRS.items():
        if not folder.exists():
            continue
        for fpath in sorted(folder.glob("*")):
            if fpath.is_file():
                rel_path = str(fpath.relative_to(WORK_DIR))
                h = hashlib.sha256()
                with open(fpath, "rb") as bf:
                    while chunk := bf.read(1024 * 1024):
                        h.update(chunk)
                info = {
                    "size_mb": round(fpath.stat().st_size / (1024 * 1024), 2),
                    "sha256": h.hexdigest()[:16],
                }
                if fpath.suffix == ".npy":
                    arr = np.load(fpath, mmap_mode="r")
                    info["shape"] = list(arr.shape)
                    info["dtype"] = str(arr.dtype)
                    sample_norms = np.linalg.norm(arr[:10], axis=1)
                    assert np.allclose(sample_norms, 1.0, atol=1e-4), f"L2 norm check failed on {rel_path}"
                manifest[rel_path] = info

    manifest_path = WORK_DIR / "cache_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    core_npy_count = sum(1 for k in manifest if k.startswith("02_embeddings_core_480_mean/") and k.endswith(".npy"))
    print(f"Core embeddings: {core_npy_count} / 32")
    print(f"Total files: {len(manifest)} | manifest: {manifest_path}")
    assert core_npy_count == 32, f"Expected 32 core embedding files, found {core_npy_count}"

    if create_zip:
        zip_base = WORK_DIR.parent / "multi_eurlex_e5_cache_bundle"
        zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(WORK_DIR))
        print(f"Created zip archive: {zip_path}")

    return manifest


manifest_result = verify_and_package_all_outputs(create_zip=True)

[VERIFIED] Core 임베딩 .npy 파일 수: 32 / 32 (8개 언어 x 4개 스플릿)
[VERIFIED] 총 생성 파일 수: 196개 | 매니페스트: /content/drive/MyDrive/기계학습_산출물/cache_manifest.json
[SUCCESS] 팀원 공유용 단일 ZIP 압축 완료: /content/drive/MyDrive/multi_eurlex_e5_cache_bundle.zip
